# Сводное сравнение трёх аналитических подходов

Высокоуровневая сводка результатов проекта: сопоставление **операций над множествами** (`venn.ipynb`), **счётного анализа** (edgeR / CPM) и **плотностного вложения последовательностей** (`mirpy_analysis.ipynb`). Тетрадь читает готовые таблицы результатов и воспроизводит ключевые выводы без повторного тяжёлого вычисления вложения.

**Главный вывод проекта:** три методологически независимые линии сходятся на общем ядре приоритетных V-сегментов α-цепи. TRAV4-2 занимает первое место во всех трёх подходах; четыре из пяти кандидатов статьи подтверждены обеими линиями mirpy; расхождения содержательно объяснимы и уточняют биологию.

---

## 0. Загрузка результатов

Все таблицы также агрегированы в `results_tables.xlsx`.

In [ ]:
import os, json
import numpy as np, pandas as pd
DATA_DIR = os.environ.get("MIRPY_DATA_DIR", "data")
def load(name): return pd.read_csv(os.path.join(DATA_DIR, name))

threeway   = load("threeway_method_comparison.csv")
validation = load("article_candidate_validation.csv")
density_v  = load("density_g1_vsegment_agg.csv")
motifs     = load("witness_motif_consensus.csv")
contrastsA = load("contrast_A_g5_vs_g3_vagg.csv")
aaVJ       = load("aaVJ_robustness_vranking.csv")
rep_stats  = json.load(open(os.path.join(DATA_DIR, "repertoire_stats.json")))
print("loaded 6 tables + repertoire stats")

## 1. Проверка кандидатов статьи по трём линиям

Пять V-сегментов, приоритизированных исходной работой, проверяются на подтверждение счётным анализом, плотностью и наличием witness-мотива.

In [ ]:
print("Article candidate validation:")
print(validation.to_string(index=False))
n_conf = (validation["verdict"].str.startswith("confirmed")).sum()
print(f"\nConfirmed by both mirpy lines: {n_conf}/{len(validation)}")

## 2. Согласованность методов

Корреляция Спирмена между счётным и плотностным ранжированием и доля общих сегментов в верхней двадцатке.

In [ ]:
from scipy.stats import spearmanr
sub = threeway.dropna(subset=["counts_rank","density_rank"])
rho, p = spearmanr(sub["counts_rank"], sub["density_rank"])
top20_c = set(sub.nsmallest(20,"counts_rank")["v_gene"])
top20_d = set(sub.nsmallest(20,"density_rank")["v_gene"])
print(f"Spearman(counts, density) = {rho:.3f}, p = {p:.2e}")
print(f"Top-20 overlap: {len(top20_c & top20_d)}/20")
print(f"\nTRAV4-2 ranks: counts={int(sub[sub.v_gene=='TRAV4-2'].counts_rank.iloc[0])}, "
      f"density={int(sub[sub.v_gene=='TRAV4-2'].density_rank.iloc[0])}")

## 3. Репертуарная геометрия (MMD)

Сингенный перенос почти не смещает репертуар; аллогенный сдвигает его на расстояние, равное межлинейному различию.

In [ ]:
print("Mean MMD from each group to intact baseline g1:")
for g,v in rep_stats["g1_vs"].items():
    print(f"  g1 – {g}: {v}")
print(f"\nWithin-g1 variability: {rep_stats['within_group_mmd']['g1']}")
print("\nPERMANOVA:")
for k in ["by_group","by_subtype","by_source"]:
    print(f"  {k}: F={rep_stats[k]['F']}, p={rep_stats[k]['p']}")

## 4. Конвергентные мотивы CDR3 и устойчивость к J

Мотив TRAV6-6 — наиболее тесно конвергентный (низшая энтропия); включение J-сегмента не меняет ранжирование.

In [ ]:
print("Convergent CDR3 motifs (sorted by tightness):")
print(motifs.sort_values("mean_pos_entropy").to_string(index=False))
print("\naaVJ robustness (J adds little): V-ranking with J vs without J")
from scipy.stats import spearmanr as sr
rr,pp = sr(aaVJ["rank_vj"], aaVJ["rank_aaV"])
print(f"Spearman(aaVJ rank, aaV rank) = {rr:.3f}, p = {pp:.2e}")

## 5. Итоговая сводная таблица

Ранги приоритетных кандидатов по всем доступным методам.

In [ ]:
summary = threeway[threeway["in_article"]].copy()
summary = summary[["v_gene","in_article","counts_rank","density_rank","density_n_enriched","witness_top500"]]
summary = summary.sort_values("density_rank")
print(summary.to_string(index=False))
summary.to_csv("final_candidate_summary.csv", index=False)

## Заключение

- **TRAV4-2** — приоритетный кандидат, подтверждён всеми тремя линиями (первое место по плотности, 4-й ранг по обилию, высшая witness-оценка).
- **TRAV9-4, семейство TRAV7D** — подтверждены обеими линиями mirpy.
- **TRAV13-1** — переопределён как признак аллоответа (обеднён в g1 по обилию), что уточняет, а не опровергает биологию статьи.
- **TRAV6-6** — исключённый исходной работой как «неоднородный», на уровне мотивов оказывается наиболее конвергентным (`CALGDMATGGNNKLTF`).
- **Устойчивость:** включение J-сегмента не меняет выводов (ρ=0,93).

Три независимых подхода согласованно подтверждают опубликованные результаты и добавляют разрешение на уровне мотивов и репертуарной геометрии. Полная интерпретация — в `методология_mirpy.docx`.